# Task 056: nmrglue — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: 2D NMR spectroscopy reconstruction using compressed sensing

**Paper**: See repository documentation
**Repository**: https://github.com/mberg88/IHCP_FEniCS

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 37.45 dB
- **SSIM**: N/A
- **Evaluation**: NMR spectrum reconstruction from non-uniform sampling (CC=0.9756)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
nmrglue — NMR Spectrum Reconstruction Inverse Problem
======================================================
Task: Reconstruct a 2D NMR spectrum from non-uniformly sampled (NUS)
      free induction decay (FID) data.

Inverse Problem:
    Given sparsely sampled time-domain FID data S(t₁, t₂) at a subset
    of t₁ points, reconstruct the full 2D frequency-domain spectrum
    I(ω₁, ω₂).

Forward Model (nmrglue):
    I(ω₁, ω₂) = FFT₂D{ S(t₁, t₂) · W(t₁, t₂) }
    where S is the complete FID and W is the NUS sampling mask.
    nmrglue handles Bruker/Varian/Agilent data formats, apodization,
    zero-filling, phasing, and spectral processing.

Inverse Solver:
    Iterative Soft Thresholding (IST) / Compressed Sensing
    reconstruction from NUS data using L1 minimisation in
    the frequency domain.

Repo: https://github.com/jjhelmus/nmrglue
Paper: Helmus & Jaroniec (2013), J. Biomol. NMR, 55, 355–367.
       doi:10.1007/s10858-013-9718-x

Usage:
    /data/yjh/spectro_env/bin/python nmrglue_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`soft_threshold`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver — Iterative Soft Thresholding (IST)
# ═══════════════════════════════════════════════════════════
def soft_threshold(x, thresh):
    """Complex soft thresholding."""
    mag = np.abs(x)
    return np.where(mag > thresh, x * (1 - thresh / np.maximum(mag, 1e-30)), 0)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_synthetic_peaks`, `synthesize_fid`, `forward_operator`, `load_or_generate_data`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Synthetic NMR Signal Generation (using nmrglue processing)
# ═══════════════════════════════════════════════════════════
def generate_synthetic_peaks(n_peaks, seed=42):
    """
    Generate random peak parameters for a 2D NMR spectrum.

    Returns list of dicts with: freq_f1, freq_f2, lw_f1, lw_f2, amplitude, phase
    """
    rng = np.random.default_rng(seed)

    peaks = []
    for i in range(n_peaks):
        peaks.append({
            "freq_f1": rng.uniform(0.15, 0.85) * SW_F1,   # Hz
            "freq_f2": rng.uniform(0.15, 0.85) * SW_F2,   # Hz
            "lw_f1": rng.uniform(10, 50),                  # Hz linewidth
            "lw_f2": rng.uniform(15, 80),                  # Hz linewidth
            "amplitude": rng.uniform(0.5, 2.0),
            "phase": rng.uniform(-0.1, 0.1),               # small phase error
        })
    return peaks

def synthesize_fid(peaks, n_f1, n_f2, sw_f1, sw_f2):
    """
    Synthesize a 2D FID from Lorentzian peaks.

    S(t1, t2) = Σ_k A_k · exp(i·2π·ν₁_k·t1 - π·Δν₁_k·t1)
                         · exp(i·2π·ν₂_k·t2 - π·Δν₂_k·t2)
    """
    dt1 = 1.0 / sw_f1
    dt2 = 1.0 / sw_f2
    t1 = np.arange(n_f1) * dt1
    t2 = np.arange(n_f2) * dt2

    fid = np.zeros((n_f1, n_f2), dtype=complex)
    for p in peaks:
        decay_f1 = np.exp(-np.pi * p["lw_f1"] * t1)
        osc_f1 = np.exp(1j * 2 * np.pi * p["freq_f1"] * t1)
        decay_f2 = np.exp(-np.pi * p["lw_f2"] * t2)
        osc_f2 = np.exp(1j * 2 * np.pi * p["freq_f2"] * t2)
        sig_f1 = p["amplitude"] * np.exp(1j * p["phase"]) * decay_f1 * osc_f1
        sig_f2 = decay_f2 * osc_f2
        fid += np.outer(sig_f1, sig_f2)

    return fid

def forward_operator(fid_full, nus_schedule):
    """
    NUS forward operator: apply sampling mask to FID.

    Uses nmrglue's processing pipeline for apodization and
    zero-filling of the direct dimension, then masks the
    indirect dimension according to the NUS schedule.

    Parameters
    ----------
    fid_full : np.ndarray  Complete 2D FID (n_f1 × n_f2).
    nus_schedule : np.ndarray  Boolean mask of sampled t1 points.

    Returns
    -------
    fid_nus : np.ndarray  NUS-sampled FID (same shape, zeros at
                          unsampled t1 rows).
    """
    # Process direct dimension (F2) with nmrglue
    dic = ng.bruker.guess_udic({"acqus": {"SW_h": SW_F2, "SFO1": OBS_F2}}, fid_full)

    # Apply nmrglue apodization to F2 (direct dimension)
    fid_proc = ng.proc_base.em(fid_full, lb=5.0)  # 5 Hz exponential line broadening

    # Zero-fill F2 if needed
    fid_proc = ng.proc_base.zf_size(fid_proc, fid_proc.shape[0], N_F2)

    # Apply NUS mask to indirect dimension (F1)
    fid_nus = np.zeros_like(fid_proc)
    fid_nus[nus_schedule, :] = fid_proc[nus_schedule, :]

    return fid_nus

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """Generate synthetic NUS NMR data."""
    print("[DATA] Generating synthetic 2D NMR peaks ...")
    peaks = generate_synthetic_peaks(N_PEAKS, SEED)

    print(f"[DATA] Synthesising complete FID ({N_F1}×{N_F2}) ...")
    fid_full = synthesize_fid(peaks, N_F1, N_F2, SW_F1, SW_F2)

    # Ground truth spectrum (full FFT with nmrglue processing)
    fid_proc = ng.proc_base.em(fid_full, lb=5.0)
    spec_gt = fftshift(fft2(fid_proc)).real
    # Normalise
    spec_gt = spec_gt / np.abs(spec_gt).max()

    # NUS schedule (random sampling of indirect dimension)
    rng = np.random.default_rng(SEED + 1)
    n_sampled = max(int(N_F1 * NUS_FRAC), 2)
    # Always include first point
    schedule = np.zeros(N_F1, dtype=bool)
    schedule[0] = True
    chosen = rng.choice(np.arange(1, N_F1), size=n_sampled - 1, replace=False)
    schedule[chosen] = True

    print(f"[DATA] NUS schedule: {schedule.sum()}/{N_F1} points "
          f"({schedule.sum()/N_F1*100:.0f}%)")

    # Apply NUS forward operator (with noise)
    rng2 = np.random.default_rng(SEED + 2)
    noise = NOISE_STD * np.abs(fid_full).max() * (
        rng2.standard_normal(fid_full.shape) +
        1j * rng2.standard_normal(fid_full.shape)
    )
    fid_noisy = fid_full + noise
    fid_nus = forward_operator(fid_noisy, schedule)

    return fid_nus, spec_gt, fid_full, schedule

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
def reconstruct(fid_nus, schedule):
    """
    Reconstruct 2D NMR spectrum from NUS data using IST
    (Iterative Soft Thresholding).

    Uses nmrglue for processing the direct dimension (apodization,
    FFT) and IST for the indirect dimension.

    Parameters
    ----------
    fid_nus : np.ndarray   NUS-sampled FID.
    schedule : np.ndarray  Boolean NUS sampling mask.

    Returns
    -------
    spec_recon : np.ndarray  Reconstructed 2D spectrum.
    """
    print(f"[RECON] IST reconstruction ({IST_ITERATIONS} iterations) ...")

    # Process F2 (direct) dimension: apodization + FFT via nmrglue
    fid_f2 = ng.proc_base.em(fid_nus, lb=5.0)
    data_f2 = np.zeros_like(fid_f2, dtype=complex)
    for i in range(fid_f2.shape[0]):
        data_f2[i, :] = fft(fid_f2[i, :])

    # IST on F1 (indirect) dimension
    # Initial estimate: zero-filled FT
    current = data_f2.copy()

    # Initial threshold from max of zero-filled spectrum
    zf_spec = fftshift(fft2(fid_f2)).real
    thresh = 0.99 * np.abs(zf_spec).max()

    for it in range(IST_ITERATIONS):
        # Step 1: FFT along F1
        spec = np.zeros_like(current, dtype=complex)
        for j in range(current.shape[1]):
            spec[:, j] = fft(current[:, j])

        # Step 2: Soft threshold in frequency domain
        spec_thresh = soft_threshold(spec, thresh)

        # Step 3: iFFT back to time domain
        for j in range(current.shape[1]):
            current[:, j] = ifft(spec_thresh[:, j])

        # Step 4: Enforce data consistency at sampled points
        current[schedule, :] = data_f2[schedule, :]

        # Decay threshold
        thresh *= IST_THRESHOLD_DECAY

        if (it + 1) % 50 == 0:
            residual = np.linalg.norm(current[schedule, :] - data_f2[schedule, :])
            print(f"[RECON]   iter {it+1:4d}  thresh={thresh:.4e}  "
                  f"residual={residual:.4e}")

    # Final spectrum
    spec_recon = np.zeros_like(current, dtype=complex)
    for j in range(current.shape[1]):
        spec_recon[:, j] = fft(current[:, j])

    spec_recon = fftshift(spec_recon).real
    spec_recon = spec_recon / np.abs(spec_recon).max()

    return spec_recon

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(spec_gt, spec_recon):
    """Compute spectrum reconstruction quality metrics."""
    # Normalise both
    gt = spec_gt / np.abs(spec_gt).max()
    rec = spec_recon / np.abs(spec_recon).max()

    # PSNR
    data_range = gt.max() - gt.min()
    mse = np.mean((gt - rec) ** 2)
    psnr = float(10 * np.log10(data_range ** 2 / max(mse, 1e-30)))

    # SSIM
    ssim_val = float(ssim_fn(gt, rec, data_range=data_range))

    # CC
    cc = float(np.corrcoef(gt.ravel(), rec.ravel())[0, 1])

    # Relative error
    re = float(np.linalg.norm(gt - rec) / max(np.linalg.norm(gt), 1e-12))

    # RMSE
    rmse = float(np.sqrt(mse))

    # Peak detection accuracy (find peaks above threshold)
    from scipy.ndimage import label
    gt_mask = gt > 0.15 * gt.max()
    rec_mask = rec > 0.15 * rec.max()
    gt_labels, n_gt = label(gt_mask)
    rec_labels, n_rec = label(rec_mask)

    metrics = {
        "PSNR": psnr,
        "SSIM": ssim_val,
        "CC": cc,
        "RE": re,
        "RMSE": rmse,
        "n_peaks_gt": int(n_gt),
        "n_peaks_recon": int(n_rec),
    }
    return metrics

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(spec_gt, spec_recon, fid_nus, schedule,
                      metrics, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    vmax = np.percentile(np.abs(spec_gt), 99)

    # (a) Ground truth spectrum
    ax = axes[0, 0]
    ax.contourf(spec_gt.T, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'(a) Ground Truth ({N_PEAKS} peaks)')
    ax.set_xlabel('F1 [pts]')
    ax.set_ylabel('F2 [pts]')

    # (b) Reconstructed spectrum
    ax = axes[0, 1]
    ax.contourf(spec_recon.T, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'(b) IST Reconstruction (NUS {NUS_FRAC*100:.0f}%)')
    ax.set_xlabel('F1 [pts]')
    ax.set_ylabel('F2 [pts]')

    # (c) NUS schedule
    ax = axes[1, 0]
    ax.stem(np.where(schedule)[0], np.ones(schedule.sum()),
            linefmt='b-', markerfmt='b.', basefmt='k-')
    ax.set_xlim(0, N_F1)
    ax.set_xlabel('Indirect dimension index')
    ax.set_ylabel('Sampled')
    ax.set_title(f'(c) NUS Schedule ({schedule.sum()}/{N_F1})')

    # (d) 1D slice comparison
    ax = axes[1, 1]
    mid = spec_gt.shape[1] // 2
    ax.plot(spec_gt[:, mid], 'b-', lw=1.5, label='GT', alpha=0.8)
    ax.plot(spec_recon[:, mid], 'r--', lw=1.5, label='IST recon', alpha=0.8)
    ax.set_xlabel('F1 [pts]')
    ax.set_ylabel('Intensity')
    ax.set_title('(d) 1D Slice (F2 midpoint)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.suptitle(
        f"nmrglue — 2D NMR NUS Reconstruction (IST)\n"
        f"PSNR={metrics['PSNR']:.1f} dB  |  SSIM={metrics['SSIM']:.4f}  |  "
        f"CC={metrics['CC']:.4f}  |  RE={metrics['RE']:.4f}",
        fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 65)
print("  nmrglue — 2D NMR NUS Reconstruction")
print("=" * 65)

fid_nus, spec_gt, fid_full, schedule = load_or_generate_data()

print("\n[RECON] Running IST reconstruction ...")
spec_recon = reconstruct(fid_nus, schedule)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("\n[EVAL] Computing metrics ...")
metrics = compute_metrics(spec_gt, spec_recon)
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), spec_recon)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), spec_gt)

visualize_results(spec_gt, spec_recon, fid_nus, schedule,
                  metrics, os.path.join(RESULTS_DIR, "reconstruction_result.png"))

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("\n" + "=" * 65)
print("  DONE — nmrglue NMR NUS reconstruction benchmark complete")
print("=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **nmrglue**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=37.45 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/mberg88/IHCP_FEniCS
